# Курсовой проект по ТОБД

**Тема:** «Анализ данных о поездках такси для выявления факторов, влияющих на длительность поездок и временных паттернов».

## 2.2. Загрузка источников и первичная проверка качества




In [1]:
pip install pandas pyarrow requests beautifulsoup4

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
import io
import pandas as pd
import requests
from bs4 import BeautifulSoup

---
## 1. Нью-Йорк — NYC Yellow Taxi Trip Records (Parquet)

Официальные данные NYC TLC за январь 2024. Загрузка файла целиком (без выборки записей).

Источник: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

In [4]:
nyc_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
nyc_df = pd.read_parquet(nyc_url)

In [5]:
nyc_df.head(5)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0


In [6]:
nyc_df.shape

(2964624, 19)

In [7]:
nyc_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2964624 entries, 0 to 2964623
Data columns (total 19 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

In [8]:
nyc_df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
VendorID,2964624.0,NaN,NaN,NaN,1.754204,1.0,2.0,2.0,2.0,6.0,0.43259
tpep_pickup_datetime,2964624,NaN,NaN,NaN,2024-01-17 00:46:36.431092,2002-12-31 22:59:39,2024-01-09 15:59:19.750000,2024-01-17 10:45:37.500000,2024-01-24 18:23:52.250000,2024-02-01 00:01:15,NaN
tpep_dropoff_datetime,2964624,NaN,NaN,NaN,2024-01-17 01:02:13.208130,2002-12-31 23:05:41,2024-01-09 16:16:23,2024-01-17 11:03:51.500000,2024-01-24 18:40:29,2024-02-02 13:56:52,NaN
passenger_count,2824462.0,NaN,NaN,NaN,1.339281,0.0,1.0,1.0,1.0,9.0,0.850282
trip_distance,2964624.0,NaN,NaN,NaN,3.652169,0.0,1.0,1.68,3.11,312722.3,225.462572
RatecodeID,2824462.0,NaN,NaN,NaN,2.069359,1.0,1.0,1.0,1.0,99.0,9.823219
store_and_fwd_flag,2824462,2,N,2813126,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PULocationID,2964624.0,NaN,NaN,NaN,166.017884,1.0,132.0,162.0,234.0,265.0,63.623914
DOLocationID,2964624.0,NaN,NaN,NaN,165.116712,1.0,114.0,162.0,234.0,265.0,69.31535
payment_type,2964624.0,NaN,NaN,NaN,1.161271,0.0,1.0,1.0,1.0,4.0,0.580869


**Количество пропущенных значений по столбцам:**

In [9]:
nyc_df.isna().sum().to_frame("missing_count")

,missing_count
VendorID,0
tpep_pickup_datetime,0
tpep_dropoff_datetime,0
passenger_count,140162
trip_distance,0
RatecodeID,140162
store_and_fwd_flag,140162
PULocationID,0
DOLocationID,0
payment_type,0


**Число полностью совпадающих строк:**

In [10]:
nyc_df.duplicated().sum()

np.int64(0)

---
## 3. Нью-Йорк — Open-Meteo Historical Weather API (JSON)

Исторические почасовые данные за январь 2024. Координаты — приблизительный центр Манхэттена, а не погода на каждом маршруте. Временная зона — `America/New_York`.

Источник: https://open-meteo.com/en/docs/historical-weather-api

In [24]:
weather_url = "https://archive-api.open-meteo.com/v1/archive"
weather_nyc_params = {
    "latitude": 40.758,
    "longitude": -73.9855,
    "start_date": "2024-01-01",
    "end_date": "2024-01-31",
    "hourly": "temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,wind_speed_10m,weather_code",
    "timezone": "America/New_York",
}
weather_nyc_response = requests.get(weather_url, params=weather_nyc_params, timeout=120)
weather_nyc_response.raise_for_status()

In [25]:
weather_nyc_json = weather_nyc_response.json()
weather_nyc_df = pd.DataFrame(weather_nyc_json["hourly"])

In [26]:
weather_nyc_df.head(5)

,time,temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,wind_speed_10m,weather_code
0,2024-01-01T00:00,2.1,67,0.0,0.0,0.0,7.7,3
1,2024-01-01T01:00,1.8,71,0.0,0.0,0.0,9.0,3
2,2024-01-01T02:00,2.4,71,0.0,0.0,0.0,10.9,3
3,2024-01-01T03:00,2.4,73,0.0,0.0,0.0,10.1,3
4,2024-01-01T04:00,2.2,75,0.0,0.0,0.0,8.1,1


In [27]:
weather_nyc_df.shape

(744, 8)

In [28]:
weather_nyc_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 744 entries, 0 to 743
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   time                  744 non-null    object 
 1   temperature_2m        744 non-null    float64
 2   relative_humidity_2m  744 non-null    int64  
 3   precipitation         744 non-null    float64
 4   rain                  744 non-null    float64
 5   snowfall              744 non-null    float64
 6   wind_speed_10m        744 non-null    float64
 7   weather_code          744 non-null    int64  
dtypes: float64(5), int64(2), object(1)
memory usage: 46.6+ KB


In [29]:
weather_nyc_df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
time,744,744,2024-01-31T23:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
temperature_2m,744.0,NaN,NaN,NaN,0.512769,4.706207,-9.5,-2.825,0.6,3.5,13.7
relative_humidity_2m,744.0,NaN,NaN,NaN,71.346774,18.502938,28.0,56.75,73.0,88.0,100.0
precipitation,744.0,NaN,NaN,NaN,0.209946,0.878153,0.0,0.0,0.0,0.0,11.9
rain,744.0,NaN,NaN,NaN,0.197177,0.87786,0.0,0.0,0.0,0.0,11.9
snowfall,744.0,NaN,NaN,NaN,0.009032,0.045478,0.0,0.0,0.0,0.0,0.49
wind_speed_10m,744.0,NaN,NaN,NaN,14.078763,7.815471,0.7,8.1,12.5,19.1,55.0
weather_code,744.0,NaN,NaN,NaN,15.365591,24.517101,0.0,0.0,3.0,3.0,73.0


**Количество пропусков:**

In [30]:
weather_nyc_df.isna().sum().to_frame("missing_count")

,missing_count
time,0
temperature_2m,0
relative_humidity_2m,0
precipitation,0
rain,0
snowfall,0
wind_speed_10m,0
weather_code,0


**Дублирующиеся временные метки:**

In [31]:
weather_nyc_df.duplicated(subset=["time"]).sum()

np.int64(0)

---
## 5. NOAA Storm Events Database (CSV).

Официальный сжатый CSV NOAA за 2024 год. Для сопоставимости с текущим этапом после чтения выделяем события января 2024 года в штатах New York и Illinois.

Источник: https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/



In [17]:
noaa_url = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/StormEvents_details-ftp_v1.0_d2024_c20260728.csv.gz"
noaa_all_df = pd.read_csv(noaa_url, compression="gzip", low_memory=False)

**Выбираем данные по двум штатам за январь 2024.** Не считаем их городскими наблюдениями.

In [18]:
noaa_df = noaa_all_df.loc[
    noaa_all_df["STATE"].isin(["NEW YORK"])
    & (noaa_all_df["MONTH_NAME"] == "January")
].copy()

In [19]:
noaa_df.head(5)

,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,...,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE
314,202401,13,500,202401,13,600,187986,1154650,NEW YORK,36,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A strong area of low pressure shifted northeas...,A measured high wind gust of 60 mph occurred a...,CSV
317,202401,14,1600,202401,14,1700,188155,1155609,NEW YORK,36,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"On January 14, 2024, a strong pressure gradien...",A measured high wind gust of 58 mph occurred a...,CSV
649,202401,10,18,202401,10,48,187413,1154988,NEW YORK,36,...,0.8,SE,ARLYN OAKS,40.6678,-73.4442,40.6727,-73.4424,A large low pressure system impacted the area ...,Block Blvd closed in East Massapequa due to fl...,CSV
909,202401,6,1800,202401,7,1300,187076,1148160,NEW YORK,36,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Low pressure pushed northeast from the Tenness...,Snowfall totals varied across Orange County fr...,CSV
910,202401,6,1800,202401,7,1300,187076,1148168,NEW YORK,36,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Low pressure pushed northeast from the Tenness...,Heavy snow was reported via Cocorahs observers...,CSV


In [20]:
noaa_df.shape

(313, 51)

In [21]:
noaa_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 313 entries, 314 to 14790
Data columns (total 51 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   BEGIN_YEARMONTH     313 non-null    int64  
 1   BEGIN_DAY           313 non-null    int64  
 2   BEGIN_TIME          313 non-null    int64  
 3   END_YEARMONTH       313 non-null    int64  
 4   END_DAY             313 non-null    int64  
 5   END_TIME            313 non-null    int64  
 6   EPISODE_ID          313 non-null    int64  
 7   EVENT_ID            313 non-null    int64  
 8   STATE               313 non-null    object 
 9   STATE_FIPS          313 non-null    int64  
 10  YEAR                313 non-null    int64  
 11  MONTH_NAME          313 non-null    object 
 12  EVENT_TYPE          313 non-null    object 
 13  CZ_TYPE             313 non-null    object 
 14  CZ_FIPS             313 non-null    int64  
 15  CZ_NAME             313 non-null    object 
 16  WFO      

In [22]:
noaa_df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
BEGIN_YEARMONTH,313.0,NaN,NaN,NaN,202401.0,0.0,202401.0,202401.0,202401.0,202401.0,202401.0
BEGIN_DAY,313.0,NaN,NaN,NaN,13.233227,6.090654,6.0,9.0,13.0,15.0,28.0
BEGIN_TIME,313.0,NaN,NaN,NaN,1235.728435,658.21333,0.0,600.0,1300.0,1700.0,2356.0
END_YEARMONTH,313.0,NaN,NaN,NaN,202401.0,0.0,202401.0,202401.0,202401.0,202401.0,202401.0
END_DAY,313.0,NaN,NaN,NaN,13.84345,6.138709,7.0,10.0,14.0,16.0,29.0
END_TIME,313.0,NaN,NaN,NaN,1238.587859,629.385984,0.0,800.0,1300.0,1800.0,2330.0
EPISODE_ID,313.0,NaN,NaN,NaN,188207.434505,636.935026,186853.0,187949.0,188567.0,188602.0,188859.0
EVENT_ID,313.0,NaN,NaN,NaN,1156962.907348,4930.879276,1146891.0,1154560.0,1159164.0,1161072.0,1161994.0
STATE,313,1,NEW YORK,313,NaN,NaN,NaN,NaN,NaN,NaN,NaN
STATE_FIPS,313.0,NaN,NaN,NaN,36.0,0.0,36.0,36.0,36.0,36.0,36.0


**Количество пропусков по столбцам:**

In [23]:
noaa_df.isna().sum().to_frame("missing_count")

,missing_count
BEGIN_YEARMONTH,0
BEGIN_DAY,0
BEGIN_TIME,0
END_YEARMONTH,0
END_DAY,0
END_TIME,0
EPISODE_ID,0
EVENT_ID,0
STATE,0
STATE_FIPS,0


**Повторяющиеся идентификаторы погодных событий:**

In [24]:
noaa_df.duplicated(subset=["EVENT_ID"]).sum()

np.int64(0)

# Статьи (html)

In [26]:

urls = [
    "https://www.zr.ru/content/news/916051-kogda-lyudi-chashche-ezdyat-na-taksi/",
    "https://rskrf.ru/tips/eksperty-obyasnyayut/ot-chego-zavisyat-tseny-na-taksi/",
    "https://vc.ru/transport/57190-spros-na-taksi-v-rossii-dni-i-chasy-kogda-avtomobili-zakazyvayut-chashe-vsego"
]

articles = []

for url in urls:
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    title = soup.title.get_text(strip=True) if soup.title else None

    for tag in soup(["script", "style", "nav", "footer"]):
        tag.decompose()

    text = "\n".join(
        p.get_text(" ", strip=True)
        for p in soup.find_all("p")
        if p.get_text(strip=True)
    )

    articles.append({
        "url": url,
        "title": title,
        "text": text
    })

articles_df = pd.DataFrame(articles)

articles_df.head()


,url,title,text
0,https://www.zr.ru/content/news/916051-kogda-ly...,Когда люди чаще ездят на такси? Исследование,От дня недели и времени зависит потребность лю...
1,https://rskrf.ru/tips/eksperty-obyasnyayut/ot-...,От чего зависят цены на такси? | Эксперты объя...,«Яндекс Go» начал тестировать новую систему от...
2,https://vc.ru/transport/57190-spros-na-taksi-v...,"Спрос на такси в России: дни и часы, когда авт...",QR-код для установки наших приложений.\nМы исп...
